## Đọc notebook05 trước khi chạy

Đây là kiểm tra kỹ thuật cầu nối training→ứng dụng: reload5baseline checkpoints Phase3 (không phải5tuned), kiểm checksum/env/schema375/class order587 và input nhị phân đúng thứ tự. Dùng4ca validation chọn có chủ đích (LR đúng/sai, bất đồng, support thấp) và5vector giả lập thêm/bớt triệu chứng; không dùng làm metric tổng thể.

LR/RF/KNN/NB Top3 từ predict_proba thật, SVC chỉ label/decision score. Bảng đúng/sai có nhãn thật; bảng giả lập không ground truth. Load+integrity/warm-up tách thời gian predict7lần, không phải cam kết API latency. Evidence structured-inference-v1 đã được Colab chạy, technical_pass=True; checksum nằm trong figure_manifest. Không fit/train/tune/test.

Khác FastAPI: notebook là công cụ đối chứng5baseline; API hiện chỉ serve approved LR tunedC2, load1lần startup và áp quality gate/session qua chat. Giữ notebook như bằng chứng mapping/inference, không thay thế API. Output nhắc Phase5A chưa làm là trạng thái lịch sử. **Không chạy lại để trình bày; đọc outputs có sẵn.** Tất cả14cell gốc giữ nguyên, không có đề xuất xóa cell khoa học.

# 05 — Kiểm thử inference có cấu trúc trước Phase 5A

Dùng năm checkpoint đã audit của baseline-v1-f8a5bfdb7b4c12cd. **LR vẫn là mặc định** theo Macro F1 validation Phase4.
Notebook chỉ kiểm kỹ thuật, không fit/tune/calibrate, gọi LLM/API hay mở locked test. Không ghi artifact Drive.
Các ca chọn có chủ đích không được dùng tính metric tổng thể hoặc chọn model mới.

Chọn kernel Colab, mount Drive, Restart Kernel và Run All notebook này. Dùng source/runtime đã đồng bộ ở Phase4;
không chạy lại notebook03/04, không upload ZIP/session/model copy hoặc cài dependency ngầm.
Lúc tạo notebook chưa có output inference Colab mới; mọi bảng/số đo dưới đây chỉ xuất hiện sau khi thực chạy.
Ca giả lập được chạy có kiểm soát để kiểm độ nhạy của model, không phải chấp thuận Top3 cho người dùng hoặc thay quality gate Phase5.


In [1]:
import runpy, platform
from pathlib import Path
if platform.system() != 'Linux' or not Path('/content').is_dir():
    raise RuntimeError('Chọn Google Colab kernel; /content phải là ổ tạm của runtime.')
DRIVE_ROOT = Path('/content/drive/MyDrive/ML_Projects/disease-prediction-ml')  # Locator sinh từ configs/runtime.json
entry = DRIVE_ROOT / 'src/data/colab_runtime.py'
if not entry.is_file():
    raise FileNotFoundError(f'Không thấy {entry}. Mount Drive và Publish source bằng prepare_colab_source.ps1; không upload script riêng.')
EXPECTED_RUNTIME_PAYLOAD = globals().get('EXPECTED_RUNTIME_PAYLOAD') or input('Paste RUNTIME_PAYLOAD_SHA256 from local Publish: ').strip()
RUNTIME = runpy.run_path(str(entry))['bootstrap_runtime'](DRIVE_ROOT, phase=3, expected_payload_sha256=EXPECTED_RUNTIME_PAYLOAD)
ROOT, runtime_paths = RUNTIME.paths.project_root, RUNTIME.paths


Stage A PASS | Colab Python 3.13.18 | Drive=/content/drive/MyDrive/ML_Projects/disease-prediction-ml | runtime/Drive read-write OK
Stage B PASS | imports=/content/disease-prediction-ml-runtime | source fingerprint=fc45ec50a18222aa4bc1ab71e55081becb478df406f7ea4c46bbd74edec3298b


In [2]:
import base64, gc, gzip, hashlib, json, time, warnings
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn import config_context
from sklearn.utils.validation import check_is_fitted
from threadpoolctl import threadpool_limits
from src.models.notebook import ColabTraining
from src.models.ensemble import Phase4, class_predictions, validate_probabilities, require
from src.models.runtime import MODEL_NAMES
from src.models.training import to_csr
from src.data.loader import sha256_file

session = ColabTraining.open(RUNTIME)  # Preflight hiện có; tuyệt đối không gọi train.
phase = Phase4(session)               # Chỉ dùng verify/loader/guard hiện có, không evaluate ensemble.
try:
    verified = phase.verify()
except Exception as exc:
    display(pd.DataFrame([{"stage": "data/checkpoint integrity", "status": "FAIL", "reason": str(exc)}]))
    raise
FEATURES, CLASSES = phase.features, phase.classes
display(verified[["model", "integrity", "input_features", "retained_features", "pipeline_bytes"]])
print("375 feature / 587 nhãn; checksum/schema/mapping/row order và metrics cũ đã kiểm.")
print("Source workspace:", ROOT, "| Drive:", runtime_paths.storage_root)
CHECK_CODE_SHA256 = 'b548198c7ef4ef8098cbeeec53a8f8d3469d8001bb6ef54003a4e2191435af0b'


Stage C PASS | manifest=50308d250d874b1b653a71c31f38dece0ec9fbb4dbc9d94f68bc1461e80b8fe9 | 7 train/validation/metadata checksums OK


,Thông tin,Giá trị
0,Runtime,colab
1,Python,3.13.18
2,Scikit-learn,1.6.1
3,RAM khả dụng (GiB),10.013
4,CPU,Intel(R) Xeon(R) CPU @ 2.20GHz
5,GPU phát hiện,[]
6,Drive đọc/ghi,True


Source workspace: /content/disease-prediction-ml-runtime
Source fingerprint: fc45ec50a18222aa4bc1ab71e55081becb478df406f7ea4c46bbd74edec3298b
Dữ liệu / đầu ra Drive: /content/drive/MyDrive/ML_Projects/disease-prediction-ml
Checksum input hợp lệ; chỉ đọc train/validation, không mở CSV test.


,model,integrity,input_features,retained_features,pipeline_bytes
0,logistic_regression,PASS,375,318,1438385
1,random_forest,PASS,375,318,14420893
2,linear_svc,PASS,375,318,1463693
3,knn,PASS,375,318,652535
4,bernoulli_nb,PASS,375,318,102624


375 feature / 587 nhãn; checksum/schema/mapping/row order và metrics cũ đã kiểm.
Source workspace: /content/disease-prediction-ml-runtime | Drive: /content/drive/MyDrive/ML_Projects/disease-prediction-ml


## 1. Input chung và các ca phải bị từ chối

build_input nhận tên feature tiếng Anh đúng schema, từ chối unknown/duplicate/empty;
validate_input yêu cầu DataFrame 1×375 đúng thứ tự, dữ liệu số hữu hạn 0/1 và ít nhất một feature hiện diện.
Không fuzzy-map hoặc tự sửa tên/thứ tự. Kiểm này là hợp đồng kỹ thuật, không chứng minh đủ thông tin lâm sàng.
Bảng chỉ in tóm tắt shape và feature hiện diện, không in375 cột.


In [3]:
def validate_input(frame, feature_order):
    if len(feature_order) != 375 or len(set(feature_order)) != 375:
        raise ValueError("Schema phải có đúng375 tên duy nhất.")
    if not isinstance(frame, pd.DataFrame) or frame.shape != (1, 375):
        raise ValueError("Input phải là DataFrame 1 x 375.")
    if frame.columns.tolist() != list(feature_order):
        raise ValueError("Tên hoặc thứ tự375 cột không khớp schema.")
    if any(not pd.api.types.is_numeric_dtype(d) or pd.api.types.is_complex_dtype(d) for d in frame.dtypes):
        raise ValueError("Feature phải là dữ liệu số0/1.")
    value = frame.to_numpy(dtype=np.float64)
    if not np.isfinite(value).all() or not np.isin(value, [0, 1]).all():
        raise ValueError("Feature phải hữu hạn và chỉ nhận0/1.")
    if not value.any():
        raise ValueError("Empty input: chưa có triệu chứng hiện diện.")
    return frame.astype(np.uint8)

def build_input(present_features, feature_order):
    if not isinstance(present_features, (list, tuple)) or not present_features:
        raise ValueError("Cần danh sách feature không rỗng.")
    if not all(isinstance(f, str) for f in present_features):
        raise ValueError("Tên feature phải là chuỗi.")
    unknown = [f for f in present_features if f not in feature_order]
    if unknown:
        raise ValueError("Unknown feature: " + repr(unknown))
    if len(set(present_features)) != len(present_features):
        raise ValueError("Duplicate feature.")
    x = pd.DataFrame(np.zeros((1, len(feature_order)), dtype=np.uint8), columns=feature_order)
    x.loc[:, list(present_features)] = 1
    return validate_input(x, feature_order)

def invalid_input_checks(feature_order):
    good = build_input([feature_order[0]], feature_order)
    bad_value = good.astype(float); bad_value.iloc[0, 0] = 2
    bad_nan = good.astype(float); bad_nan.iloc[0, 0] = np.nan
    bad_text = good.astype(object); bad_text.iloc[0, 0] = "1"
    checks = {
        "unknown": lambda: build_input(["__outside_approved_schema__"], feature_order),
        "duplicate": lambda: build_input([feature_order[0]] * 2, feature_order),
        "empty_list": lambda: build_input([], feature_order),
        "wrong_shape": lambda: validate_input(pd.concat([good, good]), feature_order),
        "wrong_width": lambda: validate_input(good.iloc[:, :-1], feature_order),
        "wrong_order": lambda: validate_input(good.iloc[:, ::-1], feature_order),
        "outside_0_1": lambda: validate_input(bad_value, feature_order),
        "nan": lambda: validate_input(bad_nan, feature_order),
        "text_value": lambda: validate_input(bad_text, feature_order),
        "all_zero": lambda: validate_input(good * 0, feature_order),
    }
    rows = []
    for name, call in checks.items():
        try:
            call()
            rows.append({"case": name, "status": "FAIL", "reason": "Input sai đã bị chấp nhận"})
        except ValueError as exc:
            rows.append({"case": name, "status": "PASS", "reason": str(exc)})
    return pd.DataFrame(rows)

invalid_checks = invalid_input_checks(FEATURES)
display(invalid_checks)
require(invalid_checks["status"].eq("PASS").all(), "Có input invalid chưa bị chặn; dừng.")
example_input = build_input(["cough", "fever"], FEATURES)
display(pd.DataFrame([{"shape": str(example_input.shape), "dtype": "uint8",
                       "present": ["cough", "fever"], "schema_order": example_input.columns.tolist() == FEATURES}]))


,case,status,reason
0,unknown,PASS,Unknown feature: ['__outside_approved_schema__']
1,duplicate,PASS,Duplicate feature.
2,empty_list,PASS,Cần danh sách feature không rỗng.
3,wrong_shape,PASS,Input phải là DataFrame 1 x 375.
4,wrong_width,PASS,Input phải là DataFrame 1 x 375.
5,wrong_order,PASS,Tên hoặc thứ tự375 cột không khớp schema.
6,outside_0_1,PASS,Feature phải hữu hạn và chỉ nhận0/1.
7,nan,PASS,Feature phải hữu hạn và chỉ nhận0/1.
8,text_value,PASS,Feature phải là dữ liệu số0/1.
9,all_zero,PASS,Empty input: chưa có triệu chứng hiện diện.


,shape,dtype,present,schema_order
0,"(1, 375)",uint8,"[cough, fever]",True


## 2. Chọn tối đa4 ca validation và5 ca giả lập

Chọn tất định từ predictions đã kiểm: LR đúng, LR sai, model bất đồng và lớp có support nhỏ nhất.
Ưu tiên hàng khác nhau; nếu một hàng đại diện nhiều nhóm thì gộp lý do. Chỉ chọn vector có ít nhất1 feature
để khớp quy tắc empty input; thiếu nhóm sẽ dừng rõ ràng. Support tính trên toàn validation.
Kiểm round-trip danh sách triệu chứng →375 bit bằng chính hàng gốc. Đây là minh họa có nhãn, không là mẫu ngẫu nhiên đại diện.


In [4]:
def select_validation_cases(x, y, predictions):
    support = np.bincount(y)
    p = predictions["logistic_regression"]
    eligible = x.to_numpy().any(axis=1)
    different = np.any(np.stack(list(predictions.values())) != p, axis=0)
    masks = [("LR đúng", p == y), ("LR sai", p != y),
             ("model bất đồng", different), ("support nhỏ nhất", support[y] == support[support > 0].min())]
    chosen = {}
    for reason, mask in masks:
        ids = np.flatnonzero(mask & eligible)
        require(len(ids) > 0, "Không tìm được nhóm validation: " + reason)
        row = next((int(i) for i in ids if int(i) not in chosen), int(ids[0]))
        chosen.setdefault(row, []).append(reason)
    return [{"case_id": f"val-{i}", "row_0based": row, "reason": " / ".join(reasons),
             "truth_id": int(y[row]), "support": int(support[y[row]])}
            for i, (row, reasons) in enumerate(chosen.items(), 1)]

validation_cases = select_validation_cases(phase.x, phase.y, phase.predictions)
validation_inputs = {}
for c in validation_cases:
    source_row = phase.x.iloc[c["row_0based"]]
    c["features"] = [f for f in FEATURES if source_row[f] == 1]
    validation_inputs[c["case_id"]] = build_input(c["features"], FEATURES)
    require(np.array_equal(validation_inputs[c["case_id"]].to_numpy()[0], source_row.to_numpy()),
            "Round-trip vector validation không khớp.")
simulated_features = {
    "sim-base": ["cough", "fever", "sore throat"],
    "sim-remove-fever": ["cough", "sore throat"],
    "sim-remove-cough": ["fever", "sore throat"],
    "sim-add-nausea": ["cough", "fever", "sore throat", "nausea"],
    "sim-only-cough": ["cough"],
}
simulated_inputs = {key: build_input(value, FEATURES) for key, value in simulated_features.items()}
display(pd.DataFrame([{**c, "truth_en": CLASSES[c["truth_id"]]} for c in validation_cases]))
print("5 ca giả lập không có ground truth; không gán bệnh kỳ vọng.")


,case_id,row_0based,reason,truth_id,support,features,truth_en
0,val-1,0,LR đúng,398,114,"[depression, shortness of breath, depressive o...",panic disorder
1,val-2,3,LR sai,398,114,"[anxiety and nervousness, depression, shortnes...",panic disorder
2,val-3,1,model bất đồng,398,114,"[depression, shortness of breath, dizziness, i...",panic disorder
3,val-4,222,support nhỏ nhất,293,1,"[knee pain, problems with movement]",injury to the knee


5 ca giả lập không có ground truth; không gán bệnh kỳ vọng.


## 3. Inference tuần tự — không fit hoặc sao chép model

Dùng Phase4._load hiện có: kiểm source/library identity, RAM, checksum trước deserialize.
Kiểm lại feature_names_in_, fitted variance/estimator, class order và custom to_csr.
LR/RF/KNN/NB dùng predict_proba thật, kiểm shape/class order/finite/range/tổng hàng và argmax so Top1.
LinearSVC chỉ trả nhãn, không có xác suất/Top3 giả. Score là điểm mô hình chưa đánh giá calibration.

Mỗi model load một lần rồi giải phóng; BLAS/working_memory dùng cấu hình hiện có.
Warm-up dùng cùng một mẫu, gồm predict và predict_proba nếu hỗ trợ; benchmark7 lần sau warm-up,
đo từng phương thức riêng, cùng mẫu đã chuẩn bị. Không bao gồm xây DataFrame/hiển thị trong thời gian predict.
Load có cả kiểm integrity/schema, không gọi nó là thời gian deserialize thuần.


In [5]:
def infer_one(model, name, frame, feature_order, classes):
    frame = validate_input(frame, feature_order)
    require(np.array_equal(model.classes_, np.arange(len(classes))), "Class order không khớp mapping.")
    pred = int(class_predictions(model.predict(frame), 1, len(classes))[0])
    result = {"label_id": pred, "label_en": classes[pred], "top3": None,
              "score_kind": "label_only_no_probability"}
    if name != "linear_svc":
        probability = np.asarray(model.predict_proba(frame))
        validate_probabilities(probability, 1, model.classes_, len(classes))
        require(int(probability[0].argmax()) == pred, "Argmax probability không khớp Top1.")
        ranks = np.argsort(-probability[0], kind="stable")[:3]
        result.update(score_kind="uncalibrated_model_score",
                      top3=[{"rank": rank, "label_id": int(i), "label_en": classes[i],
                             "score": float(probability[0, i])} for rank, i in enumerate(ranks, 1)],
                      probability=probability[0])
    return result

def measure_inference(model, name, frame, feature_order, classes, repeats=7):
    frame = validate_input(frame, feature_order)
    require(repeats >= 3, "Cần ít nhất3 lần đo, không dùng một lượt làm cam kết latency.")
    started = time.perf_counter()
    reference = infer_one(model, name, frame, feature_order, classes)
    warmup_s = time.perf_counter() - started
    output = {"warmup_s": warmup_s, "repeats": repeats}
    for method in ("predict", "predict_proba"):
        if method == "predict_proba" and name == "linear_svc":
            continue
        elapsed = []
        for _ in range(repeats):
            started = time.perf_counter()
            value = getattr(model, method)(frame)
            elapsed.append(time.perf_counter() - started)
            if method == "predict":
                require(int(class_predictions(value, 1, len(classes))[0]) == reference["label_id"],
                        "Top1 thay đổi giữa các lần đo.")
            else:
                validate_probabilities(np.asarray(value), 1, model.classes_, len(classes))
                require(np.allclose(value[0], reference["probability"], rtol=1e-10, atol=1e-12),
                        "Probability thay đổi giữa các lần đo.")
        output.update({method + "_median_ms": float(np.median(elapsed) * 1000),
                       method + "_min_ms": float(min(elapsed) * 1000),
                       method + "_max_ms": float(max(elapsed) * 1000)})
    return output


In [6]:
validation_results, simulated_results, checkpoints, timings, model_issues = [], [], [], [], []
resources = session.config["resources"]
benchmark_input = validation_inputs[validation_cases[0]["case_id"]]
for name in MODEL_NAMES:
    print(name + ": load, warm-up, inference...", flush=True)
    model, caught = None, []
    status = {"model": name, "saved_integrity": "PASS", "reload_schema_environment": "FAIL", "inference": "NOT VERIFIED"}
    try:
        with phase.guard(), warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=resources["blas_threads"]), config_context(working_memory=resources["working_memory_mb"]):
            warnings.simplefilter("always")
            started = time.perf_counter()
            model = phase._load(name, trusted=True)  # Loader hiện có, chỉ checkpoint của chính dự án.
            require(list(model.named_steps) == ["variance", "csr", "estimator"], "Pipeline steps khác audit.")
            variance, estimator = model.named_steps["variance"], model.named_steps["estimator"]
            check_is_fitted(variance); check_is_fitted(estimator)
            require(list(variance.feature_names_in_) == FEATURES and int(variance.get_support().sum()) == 318,
                    "Sai feature order/mask size.")
            require(estimator.n_features_in_ == 318 and model.named_steps["csr"].func is to_csr,
                    "Estimator/custom transformer không khớp.")
            load_s = time.perf_counter() - started
            status["reload_schema_environment"] = "PASS"
            timing = measure_inference(model, name, benchmark_input, FEATURES, CLASSES)
            timing.update(model=name, load_verified_s=load_s)
            local_val, local_sim = [], []
            for c in validation_cases:
                result = infer_one(model, name, validation_inputs[c["case_id"]], FEATURES, CLASSES)
                require(result["label_id"] == int(phase.predictions[name][c["row_0based"]]),
                        f"{name}/{c['case_id']}/row{c['row_0based']}: got {result['label_en']}, expected persisted {CLASSES[int(phase.predictions[name][c['row_0based']])]}")
                result.pop("probability", None)
                local_val.append({"model": name, **c, **result,
                                  "correct": result["label_id"] == c["truth_id"]})
            base_id = None
            base_score = None
            for case_id, frame in simulated_inputs.items():
                result = infer_one(model, name, frame, FEATURES, CLASSES)
                if case_id == "sim-base":
                    base_id = result["label_id"]
                    base_score = float(result["probability"][base_id]) if "probability" in result else None
                score = float(result["probability"][base_id]) if "probability" in result else None
                result.pop("probability", None)
                local_sim.append({"case_id": case_id, "model": name, "features": simulated_features[case_id],
                                  "truth_id": None, **result, "top1_changed_vs_base": result["label_id"] != base_id,
                                  "base_label_score_delta": score - base_score if score is not None else None})
            validation_results.extend(local_val); simulated_results.extend(local_sim); timings.append(timing)
            status["inference"] = "PASS"
    except Exception as exc:
        status["inference"] = "FAIL"
        status["reason"] = str(exc)
        model_issues.append({"model": name, "error_type": type(exc).__name__, "reason": str(exc)})
    finally:
        status["warning_counts"] = dict(Counter(w.category.__name__ for w in caught))
        status["warning_messages"] = list(dict.fromkeys(str(w.message) for w in caught))
        checkpoints.append(status)
        del model
        gc.collect()
display(pd.DataFrame(checkpoints)[["model", "saved_integrity", "reload_schema_environment", "inference"]])
if model_issues:
    display(pd.DataFrame(model_issues))


logistic_regression: load, warm-up, inference...
random_forest: load, warm-up, inference...
linear_svc: load, warm-up, inference...
knn: load, warm-up, inference...
bernoulli_nb: load, warm-up, inference...


,model,saved_integrity,reload_schema_environment,inference
0,logistic_regression,PASS,PASS,PASS
1,random_forest,PASS,PASS,PASS
2,linear_svc,PASS,PASS,PASS
3,knn,PASS,PASS,PASS
4,bernoulli_nb,PASS,PASS,PASS


## 4. Bảng kết quả thực

Bảng validation: tên bệnh gốc từ mapping587, dấu đúng/sai so nhãn thật và so prediction đã audit.
Bảng giả lập: **không có ground truth**, Δ chỉ là đổi Top1 so sim-base, không là cải thiện accuracy.
Trong bảng Top3, delta score là thay đổi score của **nhãn Top1 sim-base**, không phải xác suất bệnh;
thêm/bớt feature có thể làm đổi score dù Top1 giữ nguyên. LinearSVC không có bảng probability.
Model chưa qua kiểm tra được ghi NOT VERIFIED, không điền nhãn/score giả.


In [7]:
def show_top3(records):
    rows = []
    for r in records:
        if r["top3"] is not None:
            rows.append({"case_id": r["case_id"], "model": r["model"],
                         "Top3 / score thực": " | ".join(f'{p["rank"]}. {p["label_en"]}: {p["score"]:.6f}' for p in r["top3"]),
                         "delta_score_nhãn_base": r.get("base_label_score_delta")})
    return pd.DataFrame(rows)

lookup_val = {(r["case_id"], r["model"]): r for r in validation_results}
validation_table = []
for c in validation_cases:
    row = {"case_id": c["case_id"], "lý_do_chọn": c["reason"], "support": c["support"], "nhãn_thật": CLASSES[c["truth_id"]]}
    for name in MODEL_NAMES:
        r = lookup_val.get((c["case_id"], name))
        row[name] = (r["label_en"] + (" ✓" if r["correct"] else " ✗")) if r else "NOT VERIFIED"
    validation_table.append(row)
display(Markdown("### Validation có nhãn thật — không tính metric tổng thể từ các ca này"))
display(pd.DataFrame(validation_table))
display(show_top3(validation_results))

lookup_sim = {(r["case_id"], r["model"]): r for r in simulated_results}
simulated_table = []
for case_id, features in simulated_features.items():
    row = {"case_id": case_id, "features": " + ".join(features), "ground_truth": "Không có"}
    for name in MODEL_NAMES:
        r = lookup_sim.get((case_id, name))
        row[name] = (r["label_en"] + (" [đổi]" if r["top1_changed_vs_base"] else " [giữ]")) if r else "NOT VERIFIED"
    simulated_table.append(row)
display(Markdown("### Giả lập / độ nhạy — không gán bệnh kỳ vọng"))
display(pd.DataFrame(simulated_table))
display(show_top3(simulated_results))
sensitivity = [{"model": name, "biến_thể_đổi_Top1": [
    r["case_id"] for r in simulated_results if r["model"] == name and r["top1_changed_vs_base"]]}
    for name in MODEL_NAMES if any(r["model"] == name for r in simulated_results)]
display(pd.DataFrame(sensitivity))
print("sim-only-cough bỏ hai feature; các biến thể remove/add còn lại chỉ đổi một feature.")
print("Có/không đổi nhãn và delta score là độ nhạy kỹ thuật, không chứng minh model tốt hơn hay dự đoán giả lập đúng.")


### Validation có nhãn thật — không tính metric tổng thể từ các ca này

,case_id,lý_do_chọn,support,nhãn_thật,logistic_regression,random_forest,linear_svc,knn,bernoulli_nb
0,val-1,LR đúng,114,panic disorder,panic disorder ✓,panic disorder ✓,panic disorder ✓,panic disorder ✓,panic disorder ✓
1,val-2,LR sai,114,panic disorder,anxiety ✗,anxiety ✗,panic disorder ✓,panic disorder ✓,anxiety ✗
2,val-3,model bất đồng,114,panic disorder,panic disorder ✓,cystitis ✗,panic disorder ✓,anxiety ✗,panic disorder ✓
3,val-4,support nhỏ nhất,1,injury to the knee,arthritis of the hip ✗,arthritis of the hip ✗,arthritis of the hip ✗,flat feet ✗,injury to the leg ✗


,case_id,model,Top3 / score thực,delta_score_nhãn_base
0,val-1,logistic_regression,1. panic disorder: 0.989959 | 2. anxiety: 0.00...,None
1,val-2,logistic_regression,1. anxiety: 0.523677 | 2. panic disorder: 0.45...,None
2,val-3,logistic_regression,1. panic disorder: 0.647211 | 2. mitral valve ...,None
3,val-4,logistic_regression,1. arthritis of the hip: 0.148108 | 2. myositi...,None
4,val-1,random_forest,1. panic disorder: 0.037925 | 2. anxiety: 0.01...,None
5,val-2,random_forest,1. anxiety: 0.023177 | 2. panic disorder: 0.01...,None
6,val-3,random_forest,1. cystitis: 0.007008 | 2. anxiety: 0.006791 |...,None
7,val-4,random_forest,1. arthritis of the hip: 0.025070 | 2. injury ...,None
8,val-1,knn,1. panic disorder: 1.000000 | 2. abdominal aor...,None
9,val-2,knn,1. panic disorder: 0.800000 | 2. anxiety: 0.20...,None


### Giả lập / độ nhạy — không gán bệnh kỳ vọng

,case_id,features,ground_truth,logistic_regression,random_forest,linear_svc,knn,bernoulli_nb
0,sim-base,cough + fever + sore throat,Không có,oral mucosal lesion [giữ],cystitis [giữ],oral mucosal lesion [giữ],abscess of nose [giữ],acute sinusitis [giữ]
1,sim-remove-fever,cough + sore throat,Không có,esophagitis [đổi],cystitis [giữ],esophagitis [đổi],acute bronchospasm [đổi],acute sinusitis [giữ]
2,sim-remove-cough,fever + sore throat,Không có,conjunctivitis [đổi],cystitis [giữ],conjunctivitis [đổi],abscess of the pharynx [đổi],acute sinusitis [giữ]
3,sim-add-nausea,cough + fever + sore throat + nausea,Không có,flu [đổi],flu [đổi],flu [đổi],flu [đổi],flu [đổi]
4,sim-only-cough,cough,Không có,eczema [đổi],cystitis [giữ],eczema [đổi],teething syndrome [đổi],heart failure [đổi]


,case_id,model,Top3 / score thực,delta_score_nhãn_base
0,sim-base,logistic_regression,1. oral mucosal lesion: 0.080106 | 2. conjunct...,0.000000e+00
1,sim-remove-fever,logistic_regression,1. esophagitis: 0.056936 | 2. oral mucosal les...,-3.055463e-02
2,sim-remove-cough,logistic_regression,1. conjunctivitis: 0.055955 | 2. flu: 0.045340...,-4.703313e-02
3,sim-add-nausea,logistic_regression,1. flu: 0.554936 | 2. conjunctivitis: 0.035631...,-4.510254e-02
4,sim-only-cough,logistic_regression,1. eczema: 0.024462 | 2. oral mucosal lesion: ...,-6.206910e-02
5,sim-base,random_forest,1. cystitis: 0.007053 | 2. strep throat: 0.006...,0.000000e+00
6,sim-remove-fever,random_forest,1. cystitis: 0.007053 | 2. strep throat: 0.006...,-1.734723e-18
7,sim-remove-cough,random_forest,1. cystitis: 0.007053 | 2. strep throat: 0.006...,-1.734723e-18
8,sim-add-nausea,random_forest,1. flu: 0.010714 | 2. esophagitis: 0.008444 | ...,-1.026309e-04
9,sim-only-cough,random_forest,1. cystitis: 0.007102 | 2. esophagitis: 0.0063...,4.863600e-05


,model,biến_thể_đổi_Top1
0,logistic_regression,"[sim-remove-fever, sim-remove-cough, sim-add-n..."
1,random_forest,[sim-add-nausea]
2,linear_svc,"[sim-remove-fever, sim-remove-cough, sim-add-n..."
3,knn,"[sim-remove-fever, sim-remove-cough, sim-add-n..."
4,bernoulli_nb,"[sim-add-nausea, sim-only-cough]"


sim-only-cough bỏ hai feature; các biến thể remove/add còn lại chỉ đổi một feature.
Có/không đổi nhãn và delta score là độ nhạy kỹ thuật, không chứng minh model tốt hơn hay dự đoán giả lập đúng.


## 5. Thời gian, vấn đề còn lại và evidence

Load+integrity và warm-up là lần đo đơn, báo tách riêng. Predict/proba là median/min/max7 lần sau warm-up trên cùng mẫu.
Không bao gồm mạng/REST/LLM hay nhiều người dùng đồng thời; không dùng số này làm cam kết latency.
Warning được gộp theo loại; packet giữ thông điệp duy nhất để không in hàng trăm dòng.

PASS chỉ nói về phạm vi kỹ thuật đã chạy; không xác minh clinical quality gate hoặc triển khai Phase5A.
Nếu model/environment/checksum/input/proba không đạt, bảng ghi FAIL và phải xử lý trước khi tích hợp;
không thay model mặc định hoặc fallback fit. Copy Cell Output hai dòng INFERENCE_* để đối chiếu/log kết quả Colab thật.


In [8]:
display(pd.DataFrame(timings).round(4))
display(pd.DataFrame([{"model": r["model"], "warnings": r["warning_counts"]} for r in checkpoints]))
technical_pass = len(checkpoints) == 5 and all(r["inference"] == "PASS" and r["reload_schema_environment"] == "PASS" for r in checkpoints)
print("Kết quả kỹ thuật:", "PASS" if technical_pass else "FAIL / cần xử lý các vấn đề đã liệt kê")
print("Trước Phase5A: validator/quality gate, annotation người duyệt và hợp đồng API vẫn chưa được triển khai/đánh giá.")
print("LR mặc định giữ nguyên. Không suy chất lượng tổng thể từ các ví dụ chọn có chủ đích.")
inference_result = {
    "protocol": "structured-inference-v1", "code_sha256": CHECK_CODE_SHA256,
    "runtime": runtime_paths.runtime, "experiment": phase.experiment,
    "environment": phase.current["versions"], "feature_count": len(FEATURES), "class_count": len(CLASSES),
    "feature_schema_identity": phase.identity["feature_schema_identity"],
    "label_mapping_identity": phase.identity["label_mapping_identity"],
    "processed_fingerprint": phase.identity["processed_fingerprint"],
    "pipeline_sha256": {n: phase.metadata[n]["pipeline_sha256"] for n in MODEL_NAMES},
    "invalid_input_checks": invalid_checks.to_dict("records"), "validation_cases": validation_cases,
    "input_roundtrip": True, "checkpoints": checkpoints, "validation_results": validation_results,
    "simulated_results": simulated_results, "timings": timings, "model_issues": model_issues,
    "guards": phase.guards, "technical_pass": technical_pass, "default_model": "logistic_regression",
    "fit_calls": 0, "test_payload_opened": False, "artifacts_overwritten": False,
}
raw = json.dumps(inference_result, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")
print("INFERENCE_SHA256=" + hashlib.sha256(raw).hexdigest())
print("INFERENCE_GZIP_BASE64=" + base64.b64encode(gzip.compress(raw, mtime=0)).decode())
phase.close()
print("Dừng kiểm thử inference. Không triển khai Phase5A, REST API, LLM hoặc final test.")


,warmup_s,repeats,predict_median_ms,predict_min_ms,predict_max_ms,predict_proba_median_ms,predict_proba_min_ms,predict_proba_max_ms,model,load_verified_s
0,0.0339,7,13.7886,13.4651,13.9140,14.0880,14.0008,25.2462,logistic_regression,0.0418
1,0.1127,7,50.2328,49.3256,51.3728,50.5869,49.4585,61.4010,random_forest,1.6846
2,0.0180,7,13.2447,13.1526,13.2914,NaN,NaN,NaN,linear_svc,0.0616
3,0.0727,7,21.7156,21.4277,27.2051,39.2864,22.9739,40.0864,knn,0.0486
4,0.0791,7,35.4885,33.7299,38.6021,34.8008,34.2081,36.1707,bernoulli_nb,0.0365


,model,warnings
0,logistic_regression,{}
1,random_forest,{}
2,linear_svc,{}
3,knn,{}
4,bernoulli_nb,{}


Kết quả kỹ thuật: PASS
Trước Phase5A: validator/quality gate, annotation người duyệt và hợp đồng API vẫn chưa được triển khai/đánh giá.
LR mặc định giữ nguyên. Không suy chất lượng tổng thể từ các ví dụ chọn có chủ đích.
INFERENCE_SHA256=5dacef332521d797a74c00441bb00161ff21502078ebb12426c7dd4c4be5fd01
INFERENCE_GZIP_BASE64=H4sIAAAAAAAC/+1cS48jW1L+K1ZtBqRb5rwftUMMIyGxQAysRlfWycyT5bxtZ1qZ6equezVrNCs0YoEQG0ZohAYJAQIJqXvZiP9R/4Qv0lVddjrtsqtcj0tT07dr2pmOjBPxRcQX55E/nIW6LfKQts2kuor1+7po21ieXeRh1sRvztJpTN8tqqJsm7OLX/xwVpR5rGOZxrOLsz/7w5///Oybs3mVxRn+Oasui6Yt0kkdL+vYNEVV4modZ1XIJg0EzcMklldFXZXzWLb3AppwFbMJHoHvFe31/YX3oS6L8nKSVsvu+T/88v6zOZ4QLiNp9e0vv9mrWB3KrJpP8gpatW9DpVlRxlBPmqv0bejzrnwjvkpiXVbL2ayYlMlLafQtYD4LTbO68+xCO4tPoNCkmQahDamllePepTbmKuaO4f8mMcZUy+Byl0llfOYY40liYq4VYzKoKLjnSuqQMxpKFvOwnLWT/eGyMcQfzr6rklmR4G4+NmOG6+VyvqAxijEfS/x7cd1O8cWLMznm+OPIIGnR3cLH3HT3NO9mAFt5K4WfwQ7xwyLWxa0hk9BEAuT5FT/PXdBJniU2USkXaYav5zG0yzreWUdaff/ZrWOKDJJWxs8yZoNMrcwyxbV2WQg+ZS46biILXmY+5ildyb1LbBSJlVLgkXmUPNU